In [8]:
import random
import axelrod as axl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display

TOURNAMENT_TURNS = 200
TOURNAMENT_REPETITIONS = 5
TOURNAMENT_SEED = 0


def strategy_name(strategy):
    return strategy().name if isinstance(strategy, type) else strategy.name


def make_player(strategy):
    return strategy() if isinstance(strategy, type) else strategy.__class__()


def classify_strategy(strategy):
    class ScriptedOpponent(axl.Player):
        name = "Scripted opponent"

        def __init__(self):
            super().__init__()
            self.script = [axl.Action.C, axl.Action.C, axl.Action.D, axl.Action.C, axl.Action.C]

        def strategy(self, opponent):
            return self.script[len(self.history)]

    plays = axl.Match((make_player(strategy), ScriptedOpponent()), turns=5).play()
    classifier = getattr(strategy, "classifier", {})
    return {
        "Nice": "Si" if plays[0][0] == axl.Action.C else "No",
        "Retaliatory": "Si" if plays[3][0] == axl.Action.D else "No",
        "Forgiving": "Si" if plays[2][1] == axl.Action.D and plays[3][1] == axl.Action.C and plays[4][0] == axl.Action.C else "No",
        "Clear": "Si" if classifier.get("stochastic") is False else "No",
    }


def run_official_tournament():
    random.seed(TOURNAMENT_SEED)
    np.random.seed(TOURNAMENT_SEED)
    strategies = list(axl.axelrod_first_strategies)
    players = [strategy() for strategy in strategies]
    results = axl.Tournament(players=players, turns=TOURNAMENT_TURNS, repetitions=TOURNAMENT_REPETITIONS).play(progress_bar=False)
    scores = np.asarray(results.scores).sum(axis=1)
    classifications = [classify_strategy(strategy) for strategy in strategies]
    rows = []
    number_of_opponents = len(strategies)
    for position, name in enumerate(results.ranked_names, start=1):
        player_index = results.ranking[position - 1]
        total_score = scores[player_index]
        row = {
            "Posicion": position,
            "Estrategia": name,
            "Puntaje total": total_score,
            "Puntaje promedio": total_score / (TOURNAMENT_REPETITIONS * (number_of_opponents-1)),
        }
        row.update(classifications[player_index])
        rows.append(row)
    return players, results, pd.DataFrame(rows)


players, tournament_results, ranking = run_official_tournament()
ranking_output = widgets.Output()
match_output = widgets.Output()
strategy_options = [(strategy_name(strategy), strategy) for strategy in axl.axelrod_first_strategies]
first_selector = widgets.Dropdown(options=strategy_options, value=axl.TitForTat, description="Estrategia A:")
second_selector = widgets.Dropdown(options=strategy_options, value=axl.Grudger, description="Estrategia B:")
repetition_selector = widgets.Dropdown(options=list(range(1, TOURNAMENT_REPETITIONS + 1)), value=1, description="Repeticion:")


def render_ranking():
    with ranking_output:
        clear_output(wait=True)
        display(ranking.style.hide(axis="index").format({"Puntaje total": "{:.0f}", "Puntaje promedio": "{:.2f}"}))
        figure, axis = plt.subplots(figsize=(11, 5))
        ordered = ranking.iloc[::-1]
        axis.barh(ordered["Estrategia"], ordered["Puntaje total"], color="#176b87")
        axis.set_title("Primer torneo de Axelrod: reproduccion oficial")
        axis.set_xlabel("Puntaje acumulado en 5 repeticiones")
        axis.grid(axis="x", alpha=0.25)
        figure.tight_layout()
        display(figure)
        plt.close(figure)


def render_match(change=None):
    first, second = first_selector.value, second_selector.value
    if strategy_name(first) == strategy_name(second):
        second = next(strategy for _, strategy in strategy_options if strategy_name(strategy) != strategy_name(first))
        second_selector.value = second
    seed = TOURNAMENT_SEED + repetition_selector.value - 1
    random.seed(seed)
    np.random.seed(seed)
    match = axl.Match((make_player(first), make_player(second)), turns=TOURNAMENT_TURNS)
    plays = match.play()
    scores = np.asarray(match.scores()).cumsum(axis=0)
    first_name, second_name = strategy_name(first), strategy_name(second)
    turns = np.arange(1, len(plays) + 1)
    with match_output:
        clear_output(wait=True)
        display(widgets.HTML(f"<h3>{first_name} vs {second_name}</h3><p>200 turnos | Repeticion {repetition_selector.value}</p>"))
        moves = pd.DataFrame({"Turno": turns, first_name: [play[0] for play in plays], second_name: [play[1] for play in plays], f"{first_name} acumulado": scores[:, 0], f"{second_name} acumulado": scores[:, 1]})
        display(moves.style.hide(axis="index").format({f"{first_name} acumulado": "{:.0f}", f"{second_name} acumulado": "{:.0f}"}))
        figure, axis = plt.subplots(figsize=(11, 4))
        axis.plot(turns, scores[:, 0], label=first_name, color="#176b87", linewidth=2.5)
        axis.plot(turns, scores[:, 1], label=second_name, color="#d49332", linewidth=2.5)
        axis.set_title("Puntaje acumulado por turno")
        axis.set_xlabel("Turno")
        axis.set_ylabel("Puntos")
        axis.legend(frameon=False)
        axis.grid(alpha=0.25)
        figure.tight_layout()
        display(figure)
        plt.close(figure)


first_selector.observe(render_match, names="value")
second_selector.observe(render_match, names="value")
repetition_selector.observe(render_match, names="value")
render_ranking()
render_match()
legend = widgets.HTML("<b>Clasificaciones:</b> Nice | Retaliatory | Forgiving | Clear")
tabs = widgets.Tab(children=[
    widgets.VBox([widgets.HTML("<h2>Dashboard del primer torneo de Axelrod</h2>"), widgets.HTML("15 estrategias oficiales | 200 turnos | 5 repeticiones | semilla 0"), legend, ranking_output]),
    widgets.VBox([widgets.HTML("<h2>Explorador 1v1</h2>"), widgets.HBox([first_selector, second_selector, repetition_selector]), match_output]),
])
tabs.set_title(0, "Ranking oficial")
tabs.set_title(1, "Explorador 1v1")
display(tabs)


In [5]:
class Cloud(axl.Player):
    """Estrategia simple basada en un MDP aproximado de memoria uno."""

    name = "Cloud"
    exploration_turns = 10
    update_interval = 5

    def __init__(self):
        super().__init__()
        self.reset_learning()

    def reset_learning(self):
        self.transition_counts = {}
        self.transition_model = {}
        self.last_state = None

    def reset(self):
        super().reset()
        self.reset_learning()

    def state_key(self, opponent):
        if not self.history or not opponent.history:
            return None
        return (self.history[-1], opponent.history[-1])

    def observe_transition(self, opponent):
        if self.last_state is None or not opponent.history:
            return
        response = opponent.history[-1]
        counts = self.transition_counts.setdefault(
            self.last_state,
            {axl.Action.C: 0, axl.Action.D: 0},
        )
        counts[response] += 1

    def update_model(self):
        self.transition_model = {}
        for state, counts in self.transition_counts.items():
            total = counts[axl.Action.C] + counts[axl.Action.D]
            if total:
                self.transition_model[state] = {
                    axl.Action.C: counts[axl.Action.C] / total,
                    axl.Action.D: counts[axl.Action.D] / total,
                }

    def best_action(self, state):
        probabilities = self.transition_model.get(state)
        if probabilities is None:
            return axl.Action.C
        probability_cooperation = probabilities[axl.Action.C]
        probability_defection = probabilities[axl.Action.D]
        expected_cooperate = 3 * probability_cooperation
        expected_defect = 5 * probability_cooperation + probability_defection
        return axl.Action.D if expected_defect > expected_cooperate else axl.Action.C

    def strategy(self, opponent):
        turn = len(self.history)
        self.observe_transition(opponent)
        if turn and turn % self.update_interval == 0:
            self.update_model()
        state = self.state_key(opponent)
        self.last_state = state
        if turn < self.exploration_turns:
            return axl.Action.C if turn % 2 == 0 else axl.Action.D
        return self.best_action(state)


In [6]:
class Rulito(axl.Player):
    """Estrategia que elige acciones mediante una ecuacion de Bellman aproximada."""

    name = "Rulito"
    exploration_turns = 10
    update_interval = 5
    discount_factor = 0.90

    def __init__(self):
        super().__init__()
        self.reset_learning()

    def reset_learning(self):
        self.transition_counts = {}
        self.transition_model = {}
        self.last_state = None

    def reset(self):
        super().reset()
        self.reset_learning()

    def state_key(self, opponent):
        if not self.history or not opponent.history:
            return None
        return (self.history[-1], opponent.history[-1])

    def observe_transition(self, opponent):
        if self.last_state is None or not opponent.history:
            return
        response = opponent.history[-1]
        counts = self.transition_counts.setdefault(
            self.last_state,
            {axl.Action.C: 0, axl.Action.D: 0},
        )
        counts[response] += 1

    def update_model(self):
        self.transition_model = {}
        for state, counts in self.transition_counts.items():
            total = counts[axl.Action.C] + counts[axl.Action.D]
            if total:
                self.transition_model[state] = {
                    axl.Action.C: counts[axl.Action.C] / total,
                    axl.Action.D: counts[axl.Action.D] / total,
                }

    def response_probabilities(self, state):
        probabilities = self.transition_model.get(state)
        if probabilities is None:
            return {axl.Action.C: 0.5, axl.Action.D: 0.5}
        return probabilities

    def expected_immediate_reward(self, action, probabilities):
        if action == axl.Action.C:
            return 3 * probabilities[axl.Action.C]
        return 5 * probabilities[axl.Action.C] + probabilities[axl.Action.D]

    def bellman_value(self, action, state):
        probabilities = self.response_probabilities(state)
        immediate_value = self.expected_immediate_reward(action, probabilities)
        future_values = []
        for response in (axl.Action.C, axl.Action.D):
            next_state = (action, response)
            next_probabilities = self.response_probabilities(next_state)
            cooperate_value = self.expected_immediate_reward(axl.Action.C, next_probabilities)
            defect_value = self.expected_immediate_reward(axl.Action.D, next_probabilities)
            future_values.append(max(cooperate_value, defect_value))

        expected_future_value = sum(
            probabilities[response] * future_values[index]
            for index, response in enumerate((axl.Action.C, axl.Action.D))
        )
        return immediate_value + self.discount_factor * expected_future_value

    def best_action(self, state):
        cooperate_value = self.bellman_value(axl.Action.C, state)
        defect_value = self.bellman_value(axl.Action.D, state)
        if defect_value > cooperate_value:
            return axl.Action.D
        return axl.Action.C

    def strategy(self, opponent):
        turn = len(self.history)
        self.observe_transition(opponent)
        if turn and turn % self.update_interval == 0:
            self.update_model()

        state = self.state_key(opponent)
        self.last_state = state
        if turn < self.exploration_turns:
            return axl.Action.C if turn % 2 == 0 else axl.Action.D
        return self.best_action(state)


In [7]:
# Replicacion del primer torneo agregando Cloud y a Rulito
random.seed(TOURNAMENT_SEED)
np.random.seed(TOURNAMENT_SEED)
cloud_strategies = list(axl.axelrod_first_strategies) + [Cloud]
cloud_players = [strategy() for strategy in cloud_strategies]
cloud_results = axl.Tournament(
    players=cloud_players,
    turns=TOURNAMENT_TURNS,
    repetitions=TOURNAMENT_REPETITIONS,
).play(progress_bar=False)
cloud_scores = np.asarray(cloud_results.scores).sum(axis=1)
cloud_classifications = [classify_strategy(strategy) for strategy in cloud_strategies]
cloud_rows = []
for position, name in enumerate(cloud_results.ranked_names, start=1):
    player_index = cloud_results.ranking[position - 1]
    total_score = cloud_scores[player_index]
    row = {
        "Posicion": position,
        "Estrategia": name,
        "Puntaje total": total_score,
        "Puntaje promedio": total_score / (TOURNAMENT_REPETITIONS * len(cloud_strategies)),
    }
    row.update(cloud_classifications[player_index])
    cloud_rows.append(row)

ranking = pd.DataFrame(cloud_rows)
strategy_options = [(strategy_name(strategy), strategy) for strategy in cloud_strategies]
first_selector.options = strategy_options
second_selector.options = strategy_options
render_ranking()
render_match()
with ranking_output:
    display(widgets.HTML("<h3>Cloud incluido: 16 estrategias, 200 turnos y 5 repeticiones</h3>"))


In [ ]:
print(
    "Nombre visible:", champion_results.ranked_names[-1],
    "| puntaje Rulito:", champion_scores[champion_index],
    "| lider:", champion_results.ranked_names[0],
    "| posicion:", champion_results.ranked_names.index(champion_results.ranked_names[-1]) + 1,
)

ValueError: 'Rulito' is not in list